# 과제 - PyTorch로 mini GPT 구현

이 노트북은 과제 안내서의 구현 순서를 그대로 따릅니다.

1. 환경설정
2. NSMC 데이터 준비
3. BPE 토크나이저
4. GPTDataset / InputEmbedding
5. MultiHeadAttention
6. GPTModel
7. 사전 학습 유틸리티
8. 감성 분류 미세 조정

각 단계의 `src/` TODO를 구현한 뒤, 바로 아래 pytest 셀로 해당 단계만 확인하세요.

## 1. 환경설정

Colab에서는 GitHub 저장소 URL과 GitHub Personal Access Token을 입력해 저장소를 clone하고 `src/`를 import 경로에 추가합니다.
로컬 VS Code에서는 현재 폴더를 프로젝트 루트로 보고 실행합니다.

In [ ]:
# Colab: 이 셀을 가장 먼저 실행하세요.
import os
import subprocess
import sys
from pathlib import Path


def normalize_github_url(url: str) -> str:
    """Colab 입력값을 git clone에 사용할 수 있는 https URL로 정리합니다."""
    url = url.strip()
    if not url:
        raise ValueError("GitHub 저장소 URL을 입력해야 합니다.")
    if url.startswith("github.com/"):
        url = "https://" + url
    if not url.startswith("https://"):
        raise ValueError("저장소 URL은 https://github.com/... 또는 github.com/... 형식이어야 합니다.")
    url = url.rstrip("/")
    if not url.endswith(".git"):
        url += ".git"
    return url


if "google.colab" in sys.modules:
    from getpass import getpass

    repo_url = normalize_github_url(input("GitHub 저장소 URL (예: github.com/USERNAME/gpt-lab.git): "))
    token = getpass("GitHub Personal Access Token (Private 저장소인 경우 입력, 공개 저장소면 Enter): ").strip()
    clone_url = repo_url.replace("https://", f"https://{token}@") if token else repo_url
    repo_name = Path(repo_url[:-4]).name if repo_url.endswith(".git") else Path(repo_url).name
    repo_dir = Path("/content") / repo_name

    if not repo_dir.exists():
        subprocess.run(["git", "clone", clone_url, str(repo_dir)], check=True)
        subprocess.run(["git", "remote", "set-url", "origin", repo_url], cwd=repo_dir, check=True)
    else:
        print(f"이미 clone된 저장소를 사용합니다: {repo_dir}")

    os.chdir(repo_dir)
else:
    repo_dir = Path(".").resolve()

sys.path.insert(0, str(repo_dir / "src"))
print(f"Repo: {repo_dir}")

In [ ]:
# 단계별 테스트 실행 helper
import subprocess
import sys


def run_pytest(target: str):
    cmd = [sys.executable, "-m", "pytest", target, "-v"]
    print("실행 명령:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(repo_dir), text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        print("\n아직 통과하지 못한 테스트가 있습니다. 해당 단계의 TODO를 먼저 구현하세요.")
    else:
        print("\n선택한 테스트를 통과했습니다.")
    return result.returncode

## 2. NSMC 데이터 준비

기본 데이터는 NAVER Sentiment Movie Corpus(NSMC)입니다.
`download_data.py`는 원본 TSV를 내려받고, 사전 학습용 텍스트와 감성 분류용 JSONL을 만듭니다.

In [ ]:
from pathlib import Path

try:
    import download_data

    paths = download_data.main()
except Exception as e:
    print("데이터 준비 중 문제가 생겼습니다:", e)
    print("이미 data/ 파일이 있다면 다음 셀부터 계속 진행할 수 있습니다.")

LM_TRAIN_PATH = repo_dir / "data" / "nsmc_lm_train.txt"
LM_VAL_PATH = repo_dir / "data" / "nsmc_lm_val.txt"
print("LM train exists:", LM_TRAIN_PATH.exists(), LM_TRAIN_PATH)
print("LM val exists:", LM_VAL_PATH.exists(), LM_VAL_PATH)

In [ ]:
corpus = LM_TRAIN_PATH.read_text(encoding="utf-8") if LM_TRAIN_PATH.exists() else ""
val_corpus = LM_VAL_PATH.read_text(encoding="utf-8") if LM_VAL_PATH.exists() else ""
print("train chars:", len(corpus))
print("val chars:", len(val_corpus))
print(corpus[:200])

## 3. Step 1 - BPE 토크나이저

구현 파일: `src/bpe.py`

먼저 `pytest tests/test_bpe.py -v`를 통과시키세요. 한국어를 안전하게 다루기 위해 UTF-8 byte-level BPE로 구현해야 합니다.

In [ ]:
run_pytest("tests/test_bpe.py")

In [ ]:
# BPE 구현 후 작은 말뭉치로 인코딩/디코딩 복원을 확인합니다.
try:
    from bpe import BPETokenizer

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    sample = "이 영화는 정말 좋았다! English 123"
    ids = tokenizer.encode(sample, add_bos_eos=True)
    print(ids[:20])
    print(tokenizer.decode(ids))
except NotImplementedError as e:
    print("BPE TODO 미구현:", e)

## 4. Step 2 - GPTDataset / InputEmbedding

구현 파일: `src/dataset.py`, `src/embeddings.py`

BPE가 통과한 뒤 데이터셋과 입력 임베딩을 구현합니다.

In [ ]:
run_pytest("tests/test_dataset.py")

In [ ]:
try:
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from embeddings import InputEmbedding

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    emb = InputEmbedding(vocab_size=300, emb_dim=32, context_length=32, drop_rate=0.0)
    out = emb(inp)
    print(inp.shape, tgt.shape, out.shape)
except NotImplementedError as e:
    print("Dataset/Embedding TODO 미구현:", e)

## 5. Step 3 - MultiHeadAttention

구현 파일: `src/attention.py`

Q/K/V shape, head 분리, causal mask를 차례로 확인하세요.

In [ ]:
run_pytest("tests/test_attention.py")

## 6. Step 4 - GPTModel

구현 파일: `src/model.py`

LayerNorm, GELU, FeedForward, TransformerBlock, GPTModel, `generate_text_simple` 순서로 구현합니다.

In [ ]:
run_pytest("tests/test_model.py")

In [ ]:
try:
    import torch
    from model import GPTModel

    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    x = torch.randint(0, config["vocab_size"], (2, 16))
    logits = model(x)
    print(logits.shape)
except NotImplementedError as e:
    print("Model TODO 미구현:", e)

## 7. Step 5 - 사전 학습 유틸리티

구현 파일: `src/train.py`

loss 계산, checkpoint 저장/로드, temperature/top-k 생성, `train_model`을 구현합니다.

In [ ]:
run_pytest("tests/test_train.py")

In [ ]:
# 모든 앞 단계가 구현된 뒤 한 배치 smoke test를 실행합니다.
try:
    import torch
    from bpe import BPETokenizer
    from dataset import create_dataloader
    from model import GPTModel
    from train import calc_loss_batch

    tokenizer = BPETokenizer(vocab_size=300)
    tokenizer.train(corpus[:5000])
    token_ids = tokenizer.encode(corpus[:5000])
    loader = create_dataloader(token_ids, context_length=32, batch_size=2, shuffle=False)
    inp, tgt = next(iter(loader))
    config = {
        "vocab_size": 300,
        "context_length": 32,
        "emb_dim": 32,
        "n_heads": 4,
        "n_layers": 1,
        "drop_rate": 0.0,
        "qkv_bias": False,
    }
    model = GPTModel(config)
    loss = calc_loss_batch(inp, tgt, model, torch.device("cpu"))
    loss.backward()
    print("smoke loss:", loss.item())
except NotImplementedError as e:
    print("사전 학습 TODO 미구현:", e)

## 8. Step 6 - 감성 분류 미세 조정

구현 파일: `src/finetune.py`

NSMC JSONL/TSV를 읽어 분류 Dataset을 만들고, GPT backbone 위에 classification head를 붙입니다.

In [ ]:
run_pytest("tests/test_finetune.py")

## 9. 전체 테스트와 제출 전 확인

각 단계 테스트가 모두 통과하면 마지막에 전체 테스트를 실행합니다.

In [ ]:
run_pytest("tests/")

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/mini-gpt-lab")
RUNS_ROOT = DRIVE_ROOT / "runs"
VOCAB_ROOT = DRIVE_ROOT / "vocabs"

RUNS_ROOT.mkdir(parents=True, exist_ok=True)
VOCAB_ROOT.mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT:", DRIVE_ROOT)
print("RUNS_ROOT:", RUNS_ROOT)
print("VOCAB_ROOT:", VOCAB_ROOT)

In [ ]:
from pathlib import Path
import sys
import time
import json
import importlib
import torch

SRC_DIR = Path("src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import bpe
import dataset
import model as model_module
import train as train_module

importlib.reload(bpe)
importlib.reload(dataset)
importlib.reload(model_module)
importlib.reload(train_module)

BPETokenizer = bpe.BPETokenizer
create_dataloader = dataset.create_dataloader
GPTModel = model_module.GPTModel
train_model = train_module.train_model
load_checkpoint = train_module.load_checkpoint


# ------------------------------------------------------------
# 기본 실험 설정
# ------------------------------------------------------------
SWEEP_MODE = "light"

BASE = {
    "corpus_chars": 500_000,
    "val_chars": 50_000,
    "vocab_size": 2_000,
    "context_length": 64,
    "batch_size": 8,
    "emb_dim": 128,
    "n_heads": 4,
    "n_layers": 4,
    "drop_rate": 0.1,
    "learning_rate": 3e-4,
    "weight_decay": 0.1,
    "num_epochs": 20,
    "eval_freq": 100,
    "eval_iter": 10,
    "ckpt_freq": 100,
}

# 변인 통제:
# 한 번에 하나의 하이퍼파라미터만 바꿉니다.
# 각 항목은 small / base / large 3개입니다.
SWEEP_VALUES = {
    "learning_rate": [1e-4, BASE["learning_rate"], 5e-4],
    "batch_size": [4, BASE["batch_size"], 16],
    "drop_rate": [0.0, BASE["drop_rate"], 0.5],
    "context_length": [32, BASE["context_length"], 128],
    "emb_dim": [64, BASE["emb_dim"], 192],
    "n_heads": [2, BASE["n_heads"], 8],
    "n_layers": [2, BASE["n_layers"], 6],
}

START_CONTEXT = "이 영화는"
SEED = 123


# ------------------------------------------------------------
# 데이터 읽기
# ------------------------------------------------------------
train_text_path = Path("data/nsmc_lm_train.txt")
val_text_path = Path("data/nsmc_lm_val.txt")

if not train_text_path.exists() or not val_text_path.exists():
    raise FileNotFoundError("먼저 데이터 준비 셀 또는 python download_data.py를 실행하세요.")

raw_train_corpus = train_text_path.read_text(encoding="utf-8")
raw_val_corpus = val_text_path.read_text(encoding="utf-8")

train_corpus = raw_train_corpus[:BASE["corpus_chars"]]
val_corpus = raw_val_corpus[:BASE["val_chars"]]

print("train chars:", len(train_corpus))
print("val chars:", len(val_corpus))


# ------------------------------------------------------------
# tokenizer는 모든 모델이 공통으로 씁니다.
# vocab까지 바꾸면 tokenizer 자체가 달라져서 모델 비교가 지저분해집니다.
# ------------------------------------------------------------
vocab_path = VOCAB_ROOT / (
    f"nsmc_bpe_{SWEEP_MODE}_"
    f"vocab{BASE['vocab_size']}_"
    f"chars{BASE['corpus_chars']}.json"
)

tokenizer = BPETokenizer(vocab_size=BASE["vocab_size"])

start = time.time()

if vocab_path.exists():
    tokenizer.load(vocab_path)
    print("loaded tokenizer:", vocab_path)
else:
    tokenizer.train(train_corpus)
    tokenizer.save(vocab_path)
    print("trained tokenizer:", vocab_path)

print("tokenizer time:", round(time.time() - start, 2), "sec")

sample = "이 영화는 정말 좋았다."
assert tokenizer.decode(tokenizer.encode(sample, add_bos_eos=True), skip_special=True) == sample


# ------------------------------------------------------------
# 한 변인씩 바꾸는 run 목록 생성
# ------------------------------------------------------------
def make_run_configs(base, sweep_values):
    runs = []

    for param_name, values in sweep_values.items():
        for value in values:
            cfg = dict(base)
            cfg[param_name] = value

            if cfg["emb_dim"] % cfg["n_heads"] != 0:
                print("skip invalid:", param_name, value, "emb_dim % n_heads != 0")
                continue

            label = "base" if value == base[param_name] else str(value).replace(".", "p")
            run_name = f"{SWEEP_MODE}_{param_name}_{label}"

            runs.append({
                "run_name": run_name,
                "changed_param": param_name,
                "changed_value": value,
                "setting": cfg,
            })

    return runs


RUN_CONFIGS = make_run_configs(BASE, SWEEP_VALUES)

print("num runs:", len(RUN_CONFIGS))
for item in RUN_CONFIGS:
    print(item["run_name"])


# ------------------------------------------------------------
# 학습 실행
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


def write_json(path, data):
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


for run_item in RUN_CONFIGS:
    run_name = run_item["run_name"]
    setting = run_item["setting"]
    run_dir = RUNS_ROOT / run_name
    done_path = run_dir / "DONE.json"
    failed_path = run_dir / "FAILED.json"

    if done_path.exists():
        print("skip done:", run_name)
        continue

    run_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 80)
    print("run:", run_name)
    print("changed:", run_item["changed_param"], "=", run_item["changed_value"])
    print("=" * 80)

    try:
        torch.manual_seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)

        train_ids = tokenizer.encode(train_corpus)
        val_ids = tokenizer.encode(val_corpus)

        train_loader = create_dataloader(
            token_ids=train_ids,
            context_length=setting["context_length"],
            batch_size=setting["batch_size"],
            stride=setting["context_length"],
            shuffle=True,
            drop_last=True,
            num_workers=0,
        )

        val_loader = create_dataloader(
            token_ids=val_ids,
            context_length=setting["context_length"],
            batch_size=setting["batch_size"],
            stride=setting["context_length"],
            shuffle=False,
            drop_last=False,
            num_workers=0,
        )

        model_config = {
            "vocab_size": setting["vocab_size"],
            "context_length": setting["context_length"],
            "emb_dim": setting["emb_dim"],
            "n_heads": setting["n_heads"],
            "n_layers": setting["n_layers"],
            "drop_rate": setting["drop_rate"],
            "qkv_bias": False,
        }

        model = GPTModel(model_config).to(device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=setting["learning_rate"],
            weight_decay=setting["weight_decay"],
        )

        start_epoch = 0
        global_step = 0

        resume_path = run_dir / "checkpoints" / "last.pt"
        if resume_path.exists():
            start_epoch, global_step = load_checkpoint(
                model=model,
                optimizer=optimizer,
                path=str(resume_path),
                device=device,
            )
            print("resumed:", resume_path)
            print("start_epoch:", start_epoch)
            print("global_step:", global_step)

        num_params = sum(p.numel() for p in model.parameters())

        extra_config = {
            "mode": SWEEP_MODE,
            "seed": SEED,
            "vocab_path": str(vocab_path),
            "changed_param": run_item["changed_param"],
            "changed_value": run_item["changed_value"],
            "train_chars": len(train_corpus),
            "val_chars": len(val_corpus),
            "train_tokens": len(train_ids),
            "val_tokens": len(val_ids),
            "num_params": num_params,
            **setting,
        }

        train_losses, val_losses, track_steps = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            optimizer=optimizer,
            device=device,
            num_epochs=setting["num_epochs"],
            eval_freq=setting["eval_freq"],
            eval_iter=setting["eval_iter"],
            start_context=START_CONTEXT,
            tokenizer=tokenizer,
            ckpt_freq=setting["ckpt_freq"],
            start_epoch=start_epoch,
            global_step=global_step,
            run_dir=run_dir,
            run_name=run_name,
            config=extra_config,
        )

        write_json(done_path, {
            "run_name": run_name,
            "finished_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "changed_param": run_item["changed_param"],
            "changed_value": run_item["changed_value"],
            "final_step": track_steps[-1] if track_steps else global_step,
            "final_train_loss": train_losses[-1] if train_losses else None,
            "final_val_loss": val_losses[-1] if val_losses else None,
        })

        print("done:", run_name)

    except Exception as e:
        write_json(failed_path, {
            "run_name": run_name,
            "failed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "error_type": type(e).__name__,
            "error": str(e),
        })
        print("failed:", run_name, type(e).__name__, e)
        print("다음 run으로 넘어갑니다.")
        continue

In [ ]:
from pathlib import Path
import json
import math
import matplotlib.pyplot as plt

RUNS_ROOT = Path("/content/drive/MyDrive/mini-gpt-lab/runs")


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def read_jsonl(path):
    rows = []
    if not path.exists():
        return rows

    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))

    return rows


runs = []

for run_dir in sorted(RUNS_ROOT.iterdir()):
    if not run_dir.is_dir():
        continue

    config_path = run_dir / "config.json"
    metrics_path = run_dir / "metrics.jsonl"
    summary_path = run_dir / "summary.json"
    done_path = run_dir / "DONE.json"

    if not config_path.exists() or not metrics_path.exists():
        continue

    config = read_json(config_path)
    metrics = read_jsonl(metrics_path)
    summary = read_json(summary_path) if summary_path.exists() else {}

    if not metrics:
        continue

    best_metric = min(metrics, key=lambda row: row["val_loss"])

    runs.append({
        "run_dir": run_dir,
        "run_name": run_dir.name,
        "config": config,
        "metrics": metrics,
        "summary": summary,
        "best_metric": best_metric,
        "is_done": done_path.exists(),
    })

print("loaded runs:", len(runs))
for run in runs:
    print(run["run_name"], "done:", run["is_done"], "best val:", run["best_metric"]["val_loss"])


# ------------------------------------------------------------
# 1. 모든 run의 train/val loss 곡선
# ------------------------------------------------------------
plt.figure(figsize=(14, 7))

for run in runs:
    metrics = run["metrics"]
    steps = [row["step"] for row in metrics]
    val_losses = [row["val_loss"] for row in metrics]

    plt.plot(steps, val_losses, label=run["run_name"])

plt.xlabel("step")
plt.ylabel("val loss")
plt.title("Validation Loss by Run")
plt.legend(fontsize=8)
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# 2. best val loss 막대 그래프
# ------------------------------------------------------------
names = [run["run_name"] for run in runs]
best_vals = [run["best_metric"]["val_loss"] for run in runs]

plt.figure(figsize=(14, 6))
plt.bar(names, best_vals)
plt.xticks(rotation=80)
plt.ylabel("best val loss")
plt.title("Best Validation Loss")
plt.grid(axis="y")
plt.show()


# ------------------------------------------------------------
# 3. 과적합 확인: val_loss - train_loss
# 값이 계속 커지면 train은 외우는데 val은 나빠지는 쪽일 수 있음
# ------------------------------------------------------------
plt.figure(figsize=(14, 7))

for run in runs:
    metrics = run["metrics"]
    steps = [row["step"] for row in metrics]
    gaps = [row["val_loss"] - row["train_loss"] for row in metrics]

    plt.plot(steps, gaps, label=run["run_name"])

plt.axhline(0, color="black", linewidth=1)
plt.xlabel("step")
plt.ylabel("val loss - train loss")
plt.title("Overfitting Gap")
plt.legend(fontsize=8)
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# 4. perplexity 비교
# ------------------------------------------------------------
plt.figure(figsize=(14, 7))

for run in runs:
    metrics = run["metrics"]
    steps = [row["step"] for row in metrics]
    val_ppl = [row["val_ppl"] for row in metrics]

    plt.plot(steps, val_ppl, label=run["run_name"])

plt.xlabel("step")
plt.ylabel("val perplexity")
plt.title("Validation Perplexity")
plt.legend(fontsize=8)
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# 5. 속도 비교: tokens/sec
# ------------------------------------------------------------
names = [run["run_name"] for run in runs]
speeds = []

for run in runs:
    metrics = run["metrics"]
    last_speed = metrics[-1].get("tokens_per_sec", 0)
    speeds.append(last_speed)

plt.figure(figsize=(14, 6))
plt.bar(names, speeds)
plt.xticks(rotation=80)
plt.ylabel("tokens/sec")
plt.title("Training Speed")
plt.grid(axis="y")
plt.show()


# ------------------------------------------------------------
# 6. 하이퍼파라미터별 best val loss 비교
# ------------------------------------------------------------
groups = {}

for run in runs:
    extra = run["config"].get("extra_config", {})
    param = extra.get("changed_param", "unknown")
    value = extra.get("changed_value", "unknown")
    best_val = run["best_metric"]["val_loss"]

    if param not in groups:
        groups[param] = []

    groups[param].append((value, best_val, run["run_name"]))

for param, items in groups.items():
    items = sorted(items, key=lambda x: str(x[0]))

    labels = [str(value) for value, _, _ in items]
    values = [best_val for _, best_val, _ in items]

    plt.figure(figsize=(8, 5))
    plt.plot(labels, values, marker="o")
    plt.xlabel(param)
    plt.ylabel("best val loss")
    plt.title(f"Best Validation Loss by {param}")
    plt.grid(True)
    plt.show()

In [ ]:
# ============================================================
# 11. 감성분석 fine-tuning 실행
# pretraining best.pt를 불러와 classifier head를 붙이고 학습합니다.
# ============================================================

from pathlib import Path
import sys
import json
import time
import importlib
import torch
from torch.utils.data import DataLoader

SRC_DIR = Path("src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

import bpe
import model as model_module
import train as train_module
import finetune as finetune_module

importlib.reload(bpe)
importlib.reload(model_module)
importlib.reload(train_module)
importlib.reload(finetune_module)

BPETokenizer = bpe.BPETokenizer
GPTModel = model_module.GPTModel
load_checkpoint = train_module.load_checkpoint

ReviewSentimentDataset = finetune_module.ReviewSentimentDataset
GPTForSequenceClassification = finetune_module.GPTForSequenceClassification
train_epoch_sentiment = finetune_module.train_epoch_sentiment
evaluate_sentiment = finetune_module.evaluate_sentiment


DRIVE_ROOT = Path("/content/drive/MyDrive/mini-gpt-lab")
PRETRAIN_RUNS_ROOT = DRIVE_ROOT / "runs"
FINETUNE_RUNS_ROOT = DRIVE_ROOT / "finetune_runs"

FINETUNE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# fine-tuning 설정
# ------------------------------------------------------------
FT_CONFIG = {
    "max_length": 128,
    "batch_size": 16,
    "num_epochs": 3,
    "learning_rate": 1e-4,
    "weight_decay": 0.01,
    "drop_rate": 0.1,
    "freeze_backbone": False,
    "seed": 123,
}

# 너무 오래 걸리면 처음에는 숫자를 줄이세요.
# None이면 발견된 pretraining run 전체를 fine-tuning합니다.
MAX_PRETRAIN_RUNS = None

# 특정 run만 돌리고 싶으면 이름 일부를 넣으세요.
# 예: RUN_NAME_CONTAINS = "learning_rate"
RUN_NAME_CONTAINS = None


# ------------------------------------------------------------
# helper
# ------------------------------------------------------------
def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path, data):
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def append_jsonl(path, row):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def read_jsonl(path):
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def load_sentiment_rows():
    train_path = Path("data/nsmc_sentiment_train.jsonl")
    val_path = Path("data/nsmc_sentiment_val.jsonl")
    test_path = Path("data/nsmc_sentiment_test.jsonl")

    if not train_path.exists() or not val_path.exists() or not test_path.exists():
        raise FileNotFoundError(
            "data/nsmc_sentiment_*.jsonl 파일이 없습니다. 먼저 데이터 준비 셀 또는 download_data.py를 실행하세요."
        )

    return read_jsonl(train_path), read_jsonl(val_path), read_jsonl(test_path)


def find_pretrain_runs():
    runs = []

    for run_dir in sorted(PRETRAIN_RUNS_ROOT.iterdir()):
        if not run_dir.is_dir():
            continue

        config_path = run_dir / "config.json"
        best_ckpt_path = run_dir / "checkpoints" / "best.pt"

        if not config_path.exists() or not best_ckpt_path.exists():
            continue

        if RUN_NAME_CONTAINS is not None and RUN_NAME_CONTAINS not in run_dir.name:
            continue

        runs.append({
            "run_name": run_dir.name,
            "run_dir": run_dir,
            "config_path": config_path,
            "best_ckpt_path": best_ckpt_path,
        })

    if MAX_PRETRAIN_RUNS is not None:
        runs = runs[:MAX_PRETRAIN_RUNS]

    return runs


# ------------------------------------------------------------
# 데이터 로드
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

torch.manual_seed(FT_CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(FT_CONFIG["seed"])

train_rows, val_rows, test_rows = load_sentiment_rows()

print("sentiment train:", len(train_rows))
print("sentiment val:", len(val_rows))
print("sentiment test:", len(test_rows))


pretrain_runs = find_pretrain_runs()

print("pretrain runs:", len(pretrain_runs))
for item in pretrain_runs:
    print(item["run_name"])


# ------------------------------------------------------------
# 각 pretrained 모델을 감성분류로 fine-tuning
# ------------------------------------------------------------
for item in pretrain_runs:
    pretrain_run_name = item["run_name"]
    pretrain_config = read_json(item["config_path"])

    extra_config = pretrain_config.get("extra_config", {})
    vocab_path = Path(extra_config["vocab_path"])
    model_config = pretrain_config["model_config"]

    ft_run_name = f"ft_{pretrain_run_name}"
    ft_run_dir = FINETUNE_RUNS_ROOT / ft_run_name
    ft_run_dir.mkdir(parents=True, exist_ok=True)

    done_path = ft_run_dir / "DONE.json"
    failed_path = ft_run_dir / "FAILED.json"
    metrics_path = ft_run_dir / "metrics.jsonl"
    config_path = ft_run_dir / "config.json"
    best_path = ft_run_dir / "best.pt"
    last_path = ft_run_dir / "last.pt"

    if done_path.exists():
        print("skip done:", ft_run_name)
        continue

    print("=" * 80)
    print("fine-tune:", ft_run_name)
    print("from:", item["best_ckpt_path"])
    print("=" * 80)

    try:
        tokenizer = BPETokenizer(vocab_size=model_config["vocab_size"])
        tokenizer.load(vocab_path)

        train_dataset = ReviewSentimentDataset(
            train_rows,
            tokenizer=tokenizer,
            max_length=FT_CONFIG["max_length"],
        )
        val_dataset = ReviewSentimentDataset(
            val_rows,
            tokenizer=tokenizer,
            max_length=FT_CONFIG["max_length"],
        )
        test_dataset = ReviewSentimentDataset(
            test_rows,
            tokenizer=tokenizer,
            max_length=FT_CONFIG["max_length"],
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=FT_CONFIG["batch_size"],
            shuffle=True,
            drop_last=True,
            num_workers=0,
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=FT_CONFIG["batch_size"],
            shuffle=False,
            drop_last=False,
            num_workers=0,
        )
        test_loader = DataLoader(
            test_dataset,
            batch_size=FT_CONFIG["batch_size"],
            shuffle=False,
            drop_last=False,
            num_workers=0,
        )

        backbone = GPTModel(model_config).to(device)

        # optimizer 없이 backbone 가중치만 복원합니다.
        load_checkpoint(
            model=backbone,
            optimizer=None,
            path=str(item["best_ckpt_path"]),
            device=device,
        )

        clf_model = GPTForSequenceClassification(
            gpt_model=backbone,
            num_labels=2,
            drop_rate=FT_CONFIG["drop_rate"],
        ).to(device)

        if FT_CONFIG["freeze_backbone"]:
            for param in clf_model.gpt.parameters():
                param.requires_grad = False

        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, clf_model.parameters()),
            lr=FT_CONFIG["learning_rate"],
            weight_decay=FT_CONFIG["weight_decay"],
        )

        run_config = {
            "ft_run_name": ft_run_name,
            "pretrain_run_name": pretrain_run_name,
            "pretrain_checkpoint": str(item["best_ckpt_path"]),
            "vocab_path": str(vocab_path),
            "model_config": model_config,
            "pretrain_extra_config": extra_config,
            "finetune_config": FT_CONFIG,
            "device": str(device),
            "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        write_json(config_path, run_config)

        best_val_acc = -1.0
        best_epoch = None
        start_time = time.time()

        for epoch in range(1, FT_CONFIG["num_epochs"] + 1):
            train_loss, train_acc = train_epoch_sentiment(
                model=clf_model,
                train_loader=train_loader,
                optimizer=optimizer,
                device=device,
            )

            val_loss, val_acc = evaluate_sentiment(
                model=clf_model,
                data_loader=val_loader,
                device=device,
            )

            elapsed_sec = time.time() - start_time

            row = {
                "epoch": epoch,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "elapsed_sec": elapsed_sec,
                "learning_rate": FT_CONFIG["learning_rate"],
            }
            append_jsonl(metrics_path, row)

            print(
                f"epoch {epoch:03d} | "
                f"train loss {train_loss:.4f} | "
                f"train acc {train_acc:.4f} | "
                f"val loss {val_loss:.4f} | "
                f"val acc {val_acc:.4f}"
            )

            torch.save(
                {
                    "model_state_dict": clf_model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "epoch": epoch,
                    "config": run_config,
                    "best_val_acc": best_val_acc,
                    "best_epoch": best_epoch,
                },
                last_path,
            )

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_epoch = epoch

                torch.save(
                    {
                        "model_state_dict": clf_model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "epoch": epoch,
                        "config": run_config,
                        "best_val_acc": best_val_acc,
                        "best_epoch": best_epoch,
                    },
                    best_path,
                )

                print("best fine-tune checkpoint saved:", best_path)

        # best 모델로 test 평가
        best_ckpt = torch.load(best_path, map_location=device)
        clf_model.load_state_dict(best_ckpt["model_state_dict"])

        test_loss, test_acc = evaluate_sentiment(
            model=clf_model,
            data_loader=test_loader,
            device=device,
        )

        done = {
            "ft_run_name": ft_run_name,
            "pretrain_run_name": pretrain_run_name,
            "finished_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "best_epoch": best_epoch,
            "best_val_acc": best_val_acc,
            "test_loss": test_loss,
            "test_acc": test_acc,
            "total_time_sec": time.time() - start_time,
        }
        write_json(done_path, done)

        print("done:", ft_run_name)
        print("best val acc:", best_val_acc)
        print("test acc:", test_acc)

    except Exception as e:
        write_json(failed_path, {
            "ft_run_name": ft_run_name,
            "failed_at": time.strftime("%Y-%m-%d %H:%M:%S"),
            "error_type": type(e).__name__,
            "error": str(e),
        })
        print("failed:", ft_run_name, type(e).__name__, e)
        print("다음 fine-tuning run으로 넘어갑니다.")
        continue

In [ ]:
# ============================================================
# 12. 감성분석 fine-tuning 결과 그래프
# ============================================================

from pathlib import Path
import json
import matplotlib.pyplot as plt

FINETUNE_RUNS_ROOT = Path("/content/drive/MyDrive/mini-gpt-lab/finetune_runs")


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def read_jsonl(path):
    rows = []
    if not path.exists():
        return rows

    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))

    return rows


ft_runs = []

for run_dir in sorted(FINETUNE_RUNS_ROOT.iterdir()):
    if not run_dir.is_dir():
        continue

    config_path = run_dir / "config.json"
    metrics_path = run_dir / "metrics.jsonl"
    done_path = run_dir / "DONE.json"

    if not config_path.exists() or not metrics_path.exists():
        continue

    config = read_json(config_path)
    metrics = read_jsonl(metrics_path)
    done = read_json(done_path) if done_path.exists() else {}

    if not metrics:
        continue

    best_metric = max(metrics, key=lambda row: row["val_acc"])

    ft_runs.append({
        "run_name": run_dir.name,
        "run_dir": run_dir,
        "config": config,
        "metrics": metrics,
        "done": done,
        "best_metric": best_metric,
        "is_done": done_path.exists(),
    })

print("loaded fine-tune runs:", len(ft_runs))

for run in ft_runs:
    done = run["done"]
    print(
        run["run_name"],
        "done:", run["is_done"],
        "best val acc:", run["best_metric"]["val_acc"],
        "test acc:", done.get("test_acc"),
    )


# ------------------------------------------------------------
# 1. validation accuracy 곡선
# ------------------------------------------------------------
plt.figure(figsize=(14, 7))

for run in ft_runs:
    metrics = run["metrics"]
    epochs = [row["epoch"] for row in metrics]
    val_acc = [row["val_acc"] for row in metrics]

    plt.plot(epochs, val_acc, marker="o", label=run["run_name"])

plt.xlabel("epoch")
plt.ylabel("validation accuracy")
plt.title("Fine-tuning Validation Accuracy")
plt.legend(fontsize=8)
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# 2. train/val loss 곡선
# ------------------------------------------------------------
plt.figure(figsize=(14, 7))

for run in ft_runs:
    metrics = run["metrics"]
    epochs = [row["epoch"] for row in metrics]
    val_loss = [row["val_loss"] for row in metrics]

    plt.plot(epochs, val_loss, marker="o", label=run["run_name"])

plt.xlabel("epoch")
plt.ylabel("validation loss")
plt.title("Fine-tuning Validation Loss")
plt.legend(fontsize=8)
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# 3. best validation accuracy 막대 그래프
# ------------------------------------------------------------
names = [run["run_name"] for run in ft_runs]
best_val_accs = [run["best_metric"]["val_acc"] for run in ft_runs]

plt.figure(figsize=(14, 6))
plt.bar(names, best_val_accs)
plt.xticks(rotation=80)
plt.ylabel("best validation accuracy")
plt.title("Best Validation Accuracy")
plt.grid(axis="y")
plt.show()


# ------------------------------------------------------------
# 4. test accuracy 막대 그래프
# ------------------------------------------------------------
done_runs = [run for run in ft_runs if run["is_done"] and "test_acc" in run["done"]]

names = [run["run_name"] for run in done_runs]
test_accs = [run["done"]["test_acc"] for run in done_runs]

plt.figure(figsize=(14, 6))
plt.bar(names, test_accs)
plt.xticks(rotation=80)
plt.ylabel("test accuracy")
plt.title("Test Accuracy")
plt.grid(axis="y")
plt.show()


# ------------------------------------------------------------
# 5. 과적합 확인: train acc - val acc
# 값이 커지면 train에는 맞추는데 val 일반화가 약한 것일 수 있음
# ------------------------------------------------------------
plt.figure(figsize=(14, 7))

for run in ft_runs:
    metrics = run["metrics"]
    epochs = [row["epoch"] for row in metrics]
    gaps = [row["train_acc"] - row["val_acc"] for row in metrics]

    plt.plot(epochs, gaps, marker="o", label=run["run_name"])

plt.axhline(0, color="black", linewidth=1)
plt.xlabel("epoch")
plt.ylabel("train acc - val acc")
plt.title("Fine-tuning Overfitting Gap")
plt.legend(fontsize=8)
plt.grid(True)
plt.show()


# ------------------------------------------------------------
# 6. pretraining 하이퍼파라미터별 fine-tuning 성능 비교
# ------------------------------------------------------------
groups = {}

for run in ft_runs:
    pre_extra = run["config"].get("pretrain_extra_config", {})
    param = pre_extra.get("changed_param", "unknown")
    value = pre_extra.get("changed_value", "unknown")
    best_acc = run["best_metric"]["val_acc"]

    if param not in groups:
        groups[param] = []

    groups[param].append((value, best_acc, run["run_name"]))

for param, items in groups.items():
    items = sorted(items, key=lambda x: str(x[0]))

    labels = [str(value) for value, _, _ in items]
    values = [best_acc for _, best_acc, _ in items]

    plt.figure(figsize=(8, 5))
    plt.plot(labels, values, marker="o")
    plt.xlabel(param)
    plt.ylabel("best validation accuracy")
    plt.title(f"Fine-tuning Accuracy by Pretraining {param}")
    plt.grid(True)
    plt.show()